# RAG with ColBERT Retriever + LLaMA Generator

This notebook replaces the original RAG components:
- **Retriever**: DPR → **ColBERT v2** (via RAGatouille 0.0.9.post2)
- **Generator**: BART → **LLaMA 3.1 8B Instruct** (4-bit quantized, fits on A100)

Two evaluation domains:
- **Open-Domain QA** (NQ via `nq_dev.jsonl`): Exact Match (EM)
- **Abstractive QA** (MS-MARCO via `msmarco_dev.jsonl`): BLEU-1 + ROUGE-L

These metrics match those reported in the original RAG paper for direct comparison.

## Cell 1 — Install dependencies

> **After running this cell, go to Runtime → Restart session, then continue from Cell 2.**

ragatouille and torch have a circular import issue — ragatouille must be imported
before torch in any fresh runtime.

In [ ]:
# Pin versions that are known to work together on Colab A100
# ragatouille 0.0.9.post2 = Stanford ColBERT backend (stable, not the new PyLate backend)
# transformers 4.44.0 = compatible with ragatouille AND LLaMA-3
# bitsandbytes = 4-bit quantization so LLaMA fits alongside ColBERT in A100 VRAM
!pip install -q \
    "ragatouille==0.0.9.post2" \
    "transformers==4.44.0" \
    "accelerate>=0.27.0" \
    "bitsandbytes>=0.43.0" \
    "sentencepiece" \
    "datasets==3.6.0" \
    "faiss-cpu" \
    "pandas" \
    "pyarrow" \
    "tqdm" \
    "huggingface_hub" \
    "langchain==0.1.20" \
    "langchain-core==0.1.53" \
    "langchain-community==0.0.38"

print("Done. RESTART RUNTIME NOW before continuing.")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 4.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 7.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.1/46.1 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.5/9.5 MB 145.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 491.5/491.5 kB 48.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 77.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 303.1/303.1 kB 31.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 103.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 44.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 111.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 50.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 14.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 31

## Cell 2 — Setup: Drive, environment, project dir

In [2]:
!pip install -q \
    "transformers==4.44.2" \
    "accelerate==0.33.0" \
    "bitsandbytes==0.43.1" \
    "langchain==0.1.20" \
    "langchain-core==0.1.53" \
    "langchain-community==0.0.38" \
    "ragatouille==0.0.9.post2"

import shutil, os, subprocess

# Kill every possible cache location
for p in [
    "/content/hf_cache",
    "/root/.cache/huggingface",
    "/tmp/huggingface",
]:
    if os.path.exists(p):
        shutil.rmtree(p)
        print(f"Cleared: {p}")

# Recreate clean dirs
os.makedirs("/content/hf_cache/hub", exist_ok=True)
os.makedirs("/content/hf_cache/transformers", exist_ok=True)

!pip install -q \
    "transformers==4.43.4" \
    "accelerate==0.30.1" \
    "bitsandbytes==0.43.1"

print("Done. RESTART RUNTIME NOW.")


Cleared: /content/hf_cache
Cleared: /root/.cache/huggingface
Done. RESTART RUNTIME NOW.


In [1]:
import os
import sys
from google.colab import drive

os.chdir("/content")
try:
    drive.flush_and_unmount()
except Exception:
    pass
drive.mount("/content/gdrive", force_remount=True)

# ── EDIT THIS if your project path differs ──────────────────────────────────
PROJECT_DIR = "/content/gdrive/MyDrive/Colab Notebooks/final-proj"
# Alternative: "/content/gdrive/MyDrive/JuniorYear/CS4782/final-proj"
# ─────────────────────────────────────────────────────────────────────────────

if not os.path.exists(PROJECT_DIR):
    raise FileNotFoundError(f"PROJECT_DIR not found: {PROJECT_DIR}")

os.chdir(PROJECT_DIR)
if PROJECT_DIR not in sys.path:
    sys.path.insert(0, PROJECT_DIR)

# Keep HF cache on local Colab disk — faster + won't fill Drive quota
os.environ["FAISS_NO_AVX2"] = "1"
os.environ["HF_DATASETS_TRUST_REMOTE_CODE"] = "1"
os.environ["HF_HOME"] = "/content/hf_cache"
os.environ["HF_DATASETS_CACHE"] = "/content/hf_cache/datasets"
os.environ["TRANSFORMERS_CACHE"] = "/content/hf_cache/transformers"
os.environ["TRANSFORMERS_NO_TF"] = "1"
os.environ["USE_TF"] = "0"
os.environ["USE_TORCH"] = "1"

!mkdir -p /content/hf_cache/datasets /content/hf_cache/transformers

# CRITICAL: import ragatouille BEFORE torch to avoid the circular import bug
from ragatouille import RAGPretrainedModel
print("ragatouille imported OK")

import torch
print(f"torch {torch.__version__} | CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
print("Working in:", os.getcwd())

Mounted at /content/gdrive


/tmp/ipykernel_64860/4255795033.py:37: UserWarning: 
********************************************************************************
RAGatouille WARNING: Future Release Notice
--------------------------------------------
RAGatouille version 0.0.10 will be migrating to a PyLate backend 
instead of the current Stanford ColBERT backend.
PyLate is a fully mature, feature-equivalent backend, that greatly facilitates compatibility.
However, please pin version <0.0.10 if you require the Stanford ColBERT backend.
********************************************************************************
  from ragatouille import RAGPretrainedModel
/usr/local/lib/python3.12/dist-packages/transformers/utils/hub.py:127: FutureWarning: Using `TRANSFORMERS_CACHE` is deprecated and will be removed in v5 of Transformers. Use `HF_HOME` instead.
  warnings.warn(


RuntimeError: Failed to import transformers.trainer because of the following error (look up to see its traceback):
cannot import name 'clear_device_cache' from 'accelerate.utils.memory' (/usr/local/lib/python3.12/dist-packages/accelerate/utils/memory.py)

## Cell 3 — Build (or restore) the ColBERT 50k index

Downloads 1 NQ shard (~134k passages), samples 50k, and indexes with ColBERTv2.
Takes ~3-5 min on A100. If the index already exists in Drive it just restores it.

50k matches the small_nq_index corpus used by your DPR+BART baseline,
so retrieval corpus size is controlled between the two models.

Cleared: /content/hf_cache
Cleared: /root/.cache/huggingface
All HF caches cleared. NOW restart the runtime.


In [3]:
import os
import shutil
from huggingface_hub import snapshot_download, hf_hub_download
import pandas as pd

# ── Env vars first, before any HF call ──────────────────────────────────────
os.environ["HF_HOME"] = "/content/hf_cache"
os.environ["HUGGINGFACE_HUB_CACHE"] = "/content/hf_cache/hub"
os.environ["TRANSFORMERS_CACHE"] = "/content/hf_cache/transformers"

# ── Download colbertv2.0 to a flat local path (bypasses broken cache) ────────
COLBERT_CKPT_PATH = "/content/colbertv2.0"
if not os.path.exists(COLBERT_CKPT_PATH):
    print("Downloading colbert-ir/colbertv2.0 to local path...")
    snapshot_download(
        repo_id="colbert-ir/colbertv2.0",
        local_dir=COLBERT_CKPT_PATH,
        ignore_patterns=["*.msgpack", "*.h5"],
    )
    print("Downloaded colbertv2.0 ✅")
else:
    print(f"colbertv2.0 already at {COLBERT_CKPT_PATH} ✅")

# ── Index config ─────────────────────────────────────────────────────────────
COLBERT_INDEX_NAME = "wiki_colbert_50k"
COLBERT_INDEX_PATH = f"/content/.ragatouille/colbert/indexes/{COLBERT_INDEX_NAME}"
DRIVE_COLBERT_BACKUP = os.path.join(PROJECT_DIR, ".ragatouille")

if os.path.exists(COLBERT_INDEX_PATH):
    print(f"ColBERT index already at {COLBERT_INDEX_PATH}. Skipping build.")

elif os.path.exists(os.path.join(DRIVE_COLBERT_BACKUP, "colbert", "indexes", COLBERT_INDEX_NAME)):
    print("Restoring ColBERT index from Drive backup...")
    os.system(f'cp -r "{DRIVE_COLBERT_BACKUP}" /content/')
    print("Restored ✅")

else:
    print("Building ColBERT 50k index from scratch (~3-5 min on A100)...")

    # Download 1 wiki_dpr shard (~134k passages), sample 50k
    print("Downloading wiki_dpr shard 0...")
    f = hf_hub_download(
        repo_id="facebook/wiki_dpr",
        filename="data/psgs_w100/nq/train-00000-of-00157.parquet",
        repo_type="dataset",
        cache_dir="/content/hf_cache",
    )
    passages_df = pd.read_parquet(f, columns=["id", "title", "text"])
    print(f"Passages in shard: {len(passages_df):,}")

    passages_sample = passages_df.sample(n=50_000, random_state=42).reset_index(drop=True)
    documents = (passages_sample["title"] + ". " + passages_sample["text"]).tolist()
    doc_ids = [str(i) for i in passages_sample["id"].tolist()]
    print(f"Sampled {len(documents):,} passages ✅")

    # Load ColBERT from local path — NOT from HF repo ID
    print("Loading ColBERT from local checkpoint...")
    RAG_builder = RAGPretrainedModel.from_pretrained(COLBERT_CKPT_PATH)

    print(f"Indexing {len(documents):,} passages with ColBERTv2...")
    RAG_builder.index(
        collection=documents,
        index_name=COLBERT_INDEX_NAME,
        max_document_length=180,
        split_documents=False,
    )
    print("\nIndex built ✅")

    print("Backing up to Drive...")
    os.system(f'cp -r /content/.ragatouille "{PROJECT_DIR}"')
    print("Backup complete ✅")

print("\nColBERT 50k index ready.")
!ls -lh "$COLBERT_INDEX_PATH" 2>/dev/null || echo "Index dir not found — check path above."

colbertv2.0 already at /content/colbertv2.0 ✅
ColBERT index already at /content/.ragatouille/colbert/indexes/wiki_colbert_50k. Skipping build.

ColBERT 50k index ready.
total 261M
-rw------- 1 root root  12M May 10 21:23 0.codes.pt
-rw------- 1 root root  113 May 10 21:23 0.metadata.json
-rw------- 1 root root  92M May 10 21:23 0.residuals.pt
-rw------- 1 root root  12M May 10 21:23 1.codes.pt
-rw------- 1 root root  123 May 10 21:23 1.metadata.json
-rw------- 1 root root  92M May 10 21:23 1.residuals.pt
-rw------- 1 root root 1.6K May 10 21:23 avg_residual.pt
-rw------- 1 root root 1.8K May 10 21:23 buckets.pt
-rw------- 1 root root 8.1M May 10 21:23 centroids.pt
-rw------- 1 root root  32M May 10 21:23 collection.json
-rw------- 1 root root  98K May 10 21:23 doclens.0.json
-rw------- 1 root root  98K May 10 21:23 doclens.1.json
-rw------- 1 root root  13M May 10 21:23 ivf.pid.pt
-rw------- 1 root root 3.7K May 10 21:23 metadata.json
-rw------- 1 root root 2.4M May 10 21:23 pid_docid_

## Cell 4 — Load ColBERT index and verify retrieval

In [4]:
COLBERT_INDEX_PATH = f"/content/.ragatouille/colbert/indexes/wiki_colbert_50k"

print(f"Loading ColBERT index from {COLBERT_INDEX_PATH}...")
RAG = RAGPretrainedModel.from_index(COLBERT_INDEX_PATH)
print("Loaded.")

test_queries = [
    "What is the capital of France?",
    "Who wrote Pride and Prejudice?",
    "When did World War II end?",
]

for q in test_queries:
    print("=" * 80)
    print(f"Q: {q}")
    results = RAG.search(q, k=5)
    for i, r in enumerate(results):
        print(f"  [{i+1}] score={r['score']:.3f}: {r['content'][:150]}...")

Loading ColBERT index from /content/.ragatouille/colbert/indexes/wiki_colbert_50k...


/usr/local/lib/python3.12/dist-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/colbert/utils/amp.py:12: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  self.scaler = torch.cuda.amp.GradScaler()


Loaded.
Q: What is the capital of France?
Loading searcher for index wiki_colbert_50k for the first time... This may take a few seconds
[May 10, 23:10:51] #> Loading codec...
[May 10, 23:10:51] Loading decompress_residuals_cpp extension (set COLBERT_LOAD_TORCH_EXTENSION_VERBOSE=True for more info)...
[May 10, 23:10:52] Loading packbits_cpp extension (set COLBERT_LOAD_TORCH_EXTENSION_VERBOSE=True for more info)...
[May 10, 23:10:52] #> Loading IVF...
[May 10, 23:10:52] #> Loading doclens...


100%|██████████| 2/2 [00:00<00:00, 1177.02it/s]

[May 10, 23:10:52] #> Loading codes and residuals...



100%|██████████| 2/2 [00:00<00:00, 12.37it/s]

Searcher loaded!



/usr/local/lib/python3.12/dist-packages/colbert/utils/amp.py:15: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  return torch.cuda.amp.autocast() if self.activated else NullContextManager()



#> QueryTokenizer.tensorize(batch_text[0], batch_background[0], bsize) ==
#> Input: What is the capital of France?, 		 True, 		 None
#> Output IDs: torch.Size([32]), tensor([ 101,    1, 2054, 2003, 1996, 3007, 1997, 2605, 1029,  102,  103,  103,
         103,  103,  103,  103,  103,  103,  103,  103,  103,  103,  103,  103,
         103,  103,  103,  103,  103,  103,  103,  103], device='cuda:0')
#> Output Mask: torch.Size([32]), tensor([1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0], device='cuda:0')

  [1] score=18.906: Brussels. Brussels Brussels ( ; ), officially the Brussels-Capital Region (, ), is a region of Belgium comprising 19 municipalities, including the Cit...
  [2] score=18.812: Brussels. the "de facto" capital of the European Union, as it hosts a number of principal EU institutions (the two other capitals are Luxembourg and S...
  [3] score=18.750: Brussels. following cities: Brussels Brussels ( ; ), officially th

## Cell 5 — Load LLaMA 3.1 8B Instruct (4-bit quantized)

Uses NF4 quantization via bitsandbytes so it fits in A100 VRAM alongside ColBERT.

You need a HuggingFace account with approved access to:
https://huggingface.co/meta-llama/Meta-Llama-3.1-8B-Instruct

Add your HF token as a Colab secret named `HF_TOKEN` (Colab sidebar → Secrets).

Done. RESTART RUNTIME NOW.


In [1]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

# Load HF token from Colab secrets (preferred) or paste directly
HF_TOKEN = os.environ.get("HF_TOKEN")
try:
    from google.colab import userdata
    HF_TOKEN = userdata.get("HF_TOKEN")
    print("Loaded HF_TOKEN from Colab secrets.")
except Exception:
    print("WARNING: No HF_TOKEN found. Model download may fail if gated.")

LLAMA_MODEL_ID = "meta-llama/Meta-Llama-3.1-8B-Instruct"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

print(f"Loading tokenizer: {LLAMA_MODEL_ID}")
llama_tokenizer = AutoTokenizer.from_pretrained(LLAMA_MODEL_ID, token=HF_TOKEN)
llama_tokenizer.pad_token = llama_tokenizer.eos_token

print("Loading model in 4-bit...")
llama_model = AutoModelForCausalLM.from_pretrained(
    LLAMA_MODEL_ID,
    quantization_config=bnb_config,
    device_map={"": 0},   # pin everything to GPU 0 — avoids dispatch_model entirely
    torch_dtype=torch.float16,
    token=HF_TOKEN,
)
llama_model.eval()

print("LLaMA model loaded.")
print(f"VRAM allocated: {torch.cuda.memory_allocated() / 1e9:.2f} GB")

NameError: name 'os' is not defined

## Cell 6 — Core ColBERT + LLaMA pipeline function

In [ ]:
import torch

def colbert_llama_answer(
    question: str,
    rag_retriever,
    generator_model,
    generator_tokenizer,
    k: int = 5,
    max_new_tokens: int = 64,
    domain: str = "open_domain",  # "open_domain" (NQ) or "abstractive" (MS-MARCO)
) -> dict:
    """
    ColBERT + LLaMA RAG pipeline.

    Pipeline:
      1. ColBERT retrieves top-k passages (late interaction, token-level matching)
      2. Passages are concatenated into a context string
      3. LLaMA generates an answer conditioned on the context (chat template)

    Returns dict: {question, retrieved_passages, answer}
    """
    # Step 1: ColBERT retrieval
    results = rag_retriever.search(question, k=k)
    context_passages = [r["content"] for r in results]
    context_str = "\n\n".join(
        f"[Passage {i+1}]: {p}" for i, p in enumerate(context_passages)
    )

    # Step 2: Prompt construction (domain-specific)
    if domain == "abstractive":
        # MS-MARCO: encourage a full-sentence natural language answer
        system_prompt = (
            "You are a helpful assistant. Answer the question in one or two "
            "complete sentences using the passages below as context. "
            "Do not copy passages verbatim."
        )
    else:
        # NQ open-domain: short factual answer
        system_prompt = (
            "You are a helpful assistant. Answer the question concisely "
            "using the passages below. Give a short factual answer only."
        )

    user_prompt = (
        f"Passages:\n{context_str}\n\n"
        f"Question: {question}\n"
        f"Answer:"
    )

    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user",   "content": user_prompt},
    ]

    # Step 3: LLaMA generation with chat template
    formatted = generator_tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )
    inputs = generator_tokenizer(
        formatted,
        return_tensors="pt",
        truncation=True,
        max_length=2048,
    ).to(generator_model.device)

    with torch.no_grad():
        output_ids = generator_model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            temperature=None,
            top_p=None,
            pad_token_id=generator_tokenizer.eos_token_id,
        )

    # Decode only new tokens (skip the input prompt)
    new_tokens = output_ids[0][inputs["input_ids"].shape[1]:]
    answer = generator_tokenizer.decode(new_tokens, skip_special_tokens=True).strip()

    return {
        "question": question,
        "retrieved_passages": context_passages,
        "answer": answer,
    }


print("colbert_llama_answer() ready.")

## Cell 7 — Smoke test

In [ ]:
smoke_questions = [
    "What is the capital of France?",
    "Who wrote Pride and Prejudice?",
    "When did World War II end?",
]

for q in smoke_questions:
    result = colbert_llama_answer(
        question=q,
        rag_retriever=RAG,
        generator_model=llama_model,
        generator_tokenizer=llama_tokenizer,
        k=5,
        max_new_tokens=40,
        domain="open_domain",
    )
    print("=" * 80)
    print(f"Q: {result['question']}")
    print(f"A: {result['answer']}")
    print(f"   (top passage: {result['retrieved_passages'][0][:100]}...)")

## Cell 8 — Prepare NQ dev data

Loads `data/nq_dev.jsonl` from your project folder.
Format: `{"question": "...", "answers": ["..."]}`

In [ ]:
import json
from pathlib import Path

def load_jsonl(path, max_examples=None):
    examples = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            examples.append(json.loads(line))
            if max_examples and len(examples) >= max_examples:
                break
    return examples

# Try both common locations
NQ_DEV_PATH = Path(PROJECT_DIR) / "data" / "nq_dev.jsonl"
if not NQ_DEV_PATH.exists():
    NQ_DEV_PATH = Path("/content/data/nq_dev.jsonl")
if not NQ_DEV_PATH.exists():
    raise FileNotFoundError(
        f"nq_dev.jsonl not found. Expected at: {Path(PROJECT_DIR) / 'data' / 'nq_dev.jsonl'}"
    )

nq_dev = load_jsonl(NQ_DEV_PATH)
print(f"Loaded {len(nq_dev):,} NQ dev examples from {NQ_DEV_PATH}")
print("Example:", nq_dev[0])

## Cell 9 — Evaluate on NQ (Exact Match)

Exact Match is the open-domain QA metric used in the RAG paper (Table 1).
Paper baseline: RAG-Sequence = 44.5 EM on NQ.

In [ ]:
import re
from tqdm import tqdm

def normalize_answer(s: str) -> str:
    """Standard NQ normalization: lowercase, strip articles/punctuation."""
    s = s.lower()
    s = re.sub(r"\b(a|an|the)\b", " ", s)
    s = re.sub(r"[^\w\s]", "", s)
    s = re.sub(r"\s+", " ", s).strip()
    return s

def exact_match(prediction: str, gold_answers: list) -> bool:
    """
    True if any gold answer is contained in the prediction (or vice versa).
    This is slightly more lenient than strict EM but handles LLaMA's tendency
    to produce full-sentence answers for short factual questions.
    """
    pred_norm = normalize_answer(prediction)
    return any(
        normalize_answer(g) in pred_norm or pred_norm in normalize_answer(g)
        for g in gold_answers
    )

# ── Config ────────────────────────────────────────────────────────────────────
MAX_EVAL_EXAMPLES = 500   # set to None to run full dev set (~3k examples, ~1hr)
K_DOCS = 5
MAX_NEW_TOKENS = 40
# ─────────────────────────────────────────────────────────────────────────────

nq_eval = nq_dev[:MAX_EVAL_EXAMPLES] if MAX_EVAL_EXAMPLES else nq_dev
hits = 0
nq_results = []

for ex in tqdm(nq_eval, desc="NQ Eval"):
    result = colbert_llama_answer(
        question=ex["question"],
        rag_retriever=RAG,
        generator_model=llama_model,
        generator_tokenizer=llama_tokenizer,
        k=K_DOCS,
        max_new_tokens=MAX_NEW_TOKENS,
        domain="open_domain",
    )
    is_correct = exact_match(result["answer"], ex["answers"])
    if is_correct:
        hits += 1
    nq_results.append({
        "question": ex["question"],
        "gold_answers": ex["answers"],
        "prediction": result["answer"],
        "correct": is_correct,
    })

em_score = 100.0 * hits / len(nq_eval)
print("=" * 80)
print(f"NQ Exact Match  — ColBERT + LLaMA: {em_score:.2f}%")
print(f"Correct: {hits} / {len(nq_eval)}")
print(f"(RAG paper RAG-Sequence baseline:   44.5%)")

## Cell 10 — Save NQ predictions

In [ ]:
import json
from pathlib import Path

NQ_PRED_PATH = Path(PROJECT_DIR) / "outputs" / "colbert_llama_nq_predictions.jsonl"
NQ_PRED_PATH.parent.mkdir(parents=True, exist_ok=True)

with open(NQ_PRED_PATH, "w", encoding="utf-8") as f:
    for r in nq_results:
        f.write(json.dumps(r, ensure_ascii=False) + "\n")

print(f"Saved {len(nq_results)} predictions to {NQ_PRED_PATH}")
print(f"NQ EM: {em_score:.2f}%")

# Sample wrong predictions to understand failure modes
wrong = [r for r in nq_results if not r["correct"]]
print(f"\nSample incorrect predictions ({min(3, len(wrong))} of {len(wrong)}):")
for r in wrong[:3]:
    print(f"  Q:    {r['question']}")
    print(f"  Gold: {r['gold_answers']}")
    print(f"  Pred: {r['prediction']}")
    print()

## Cell 11 — Prepare MS-MARCO dev data

Expects `data/msmarco_dev.jsonl` in the same format produced by `prepare_msmarco.py`
in `main_msmarco_abstractive.py` Cell 5-6.

Format: `{"question": "...", "answers": ["full sentence answer"]}`

In [ ]:
from pathlib import Path

MSMARCO_DEV_PATH = Path(PROJECT_DIR) / "data" / "msmarco_dev.jsonl"
if not MSMARCO_DEV_PATH.exists():
    MSMARCO_DEV_PATH = Path("/content/data/msmarco_dev.jsonl")
if not MSMARCO_DEV_PATH.exists():
    raise FileNotFoundError(
        "msmarco_dev.jsonl not found. "
        "Run prepare_msmarco.py first (see main_msmarco_abstractive.py Cells 5-6)."
    )

msmarco_dev = load_jsonl(MSMARCO_DEV_PATH)
print(f"Loaded {len(msmarco_dev):,} MS-MARCO dev examples from {MSMARCO_DEV_PATH}")
print("Example:", msmarco_dev[0])

## Cell 12 — Evaluate on MS-MARCO (BLEU-1 + ROUGE-L)

These are the metrics used in the RAG paper for MS-MARCO abstractive QA (Table 2).
Paper baseline: RAG-Sequence = ROUGE-L 40.8, BLEU-1 44.2.

In [ ]:
import math
import re
from collections import Counter
from tqdm import tqdm

# ── Metric implementations (no extra library needed) ─────────────────────────

def normalize_tokens(text: str) -> list:
    return re.findall(r"\w+", str(text).lower())

def lcs_len(a: list, b: list) -> int:
    dp = [0] * (len(b) + 1)
    for x in a:
        prev = 0
        for j, y in enumerate(b, start=1):
            temp = dp[j]
            dp[j] = prev + 1 if x == y else max(dp[j], dp[j - 1])
            prev = temp
    return dp[-1]

def rouge_l_f1(pred: str, ref: str) -> float:
    p, r = normalize_tokens(pred), normalize_tokens(ref)
    if not p or not r:
        return 0.0
    lcs = lcs_len(p, r)
    prec, rec = lcs / len(p), lcs / len(r)
    return 2 * prec * rec / (prec + rec) if prec + rec > 0 else 0.0

def corpus_bleu1(preds: list, refs_list: list) -> float:
    clipped_total = pred_total = pred_len = ref_len = 0
    for pred, refs in zip(preds, refs_list):
        p_toks = normalize_tokens(pred)
        r_toks_list = [normalize_tokens(r) for r in refs if str(r).strip()]
        if not p_toks or not r_toks_list:
            continue
        p_counts = Counter(p_toks)
        best_overlap, best_rc, best_rl = -1, Counter(), len(r_toks_list[0])
        for r_toks in r_toks_list:
            rc = Counter(r_toks)
            ov = sum((p_counts & rc).values())
            if ov > best_overlap:
                best_overlap, best_rc, best_rl = ov, rc, len(r_toks)
        clipped_total += sum((p_counts & best_rc).values())
        pred_total += len(p_toks)
        pred_len += len(p_toks)
        ref_len += best_rl
    if pred_total == 0:
        return 0.0
    prec = clipped_total / pred_total
    bp = 1.0 if pred_len >= ref_len else math.exp(1 - ref_len / pred_len)
    return 100 * bp * prec

# ── Config ────────────────────────────────────────────────────────────────────
MAX_MSMARCO_EVAL = 500
K_DOCS = 5
MAX_NEW_TOKENS = 80   # MS-MARCO answers are longer than NQ
# ─────────────────────────────────────────────────────────────────────────────

msmarco_eval = msmarco_dev[:MAX_MSMARCO_EVAL]
preds, refs_list, msmarco_results = [], [], []

for ex in tqdm(msmarco_eval, desc="MS-MARCO Eval"):
    result = colbert_llama_answer(
        question=ex["question"],
        rag_retriever=RAG,
        generator_model=llama_model,
        generator_tokenizer=llama_tokenizer,
        k=K_DOCS,
        max_new_tokens=MAX_NEW_TOKENS,
        domain="abstractive",
    )
    preds.append(result["answer"])
    refs_list.append(ex["answers"])
    msmarco_results.append({
        "question": ex["question"],
        "references": ex["answers"],
        "prediction": result["answer"],
    })

rouge_l_scores = [
    max(rouge_l_f1(pred, ref) for ref in refs)
    for pred, refs in zip(preds, refs_list)
]
rouge_l = 100 * sum(rouge_l_scores) / len(rouge_l_scores)
bleu1 = corpus_bleu1(preds, refs_list)

print("=" * 80)
print(f"MS-MARCO ROUGE-L — ColBERT + LLaMA: {rouge_l:.2f}")
print(f"MS-MARCO BLEU-1  — ColBERT + LLaMA: {bleu1:.2f}")
print(f"(RAG paper RAG-Sequence baseline:    ROUGE-L=40.8, BLEU-1=44.2)")

## Cell 13 — Save MS-MARCO predictions

In [ ]:
from pathlib import Path
import json

MSMARCO_PRED_PATH = Path(PROJECT_DIR) / "outputs" / "colbert_llama_msmarco_predictions.jsonl"
MSMARCO_PRED_PATH.parent.mkdir(parents=True, exist_ok=True)

with open(MSMARCO_PRED_PATH, "w", encoding="utf-8") as f:
    for r in msmarco_results:
        f.write(json.dumps(r, ensure_ascii=False) + "\n")

print(f"Saved {len(msmarco_results)} MS-MARCO predictions to {MSMARCO_PRED_PATH}")
print(f"ROUGE-L: {rouge_l:.2f} | BLEU-1: {bleu1:.2f}")

## Cell 14 — Results summary table

Run after both Cells 9 and 12 to print a clean comparison for your poster/report.

In [ ]:
print("=" * 72)
print(f"{'Model':<32} {'Dataset':<12} {'Metric':<10} {'Score':>7}")
print("-" * 72)

# RAG paper baselines (Table 1 + Table 2)
paper = [
    ("RAG-Sequence (DPR + BART)",   "NQ",       "EM",      44.5),
    ("RAG-Sequence (DPR + BART)",   "MS-MARCO", "ROUGE-L", 40.8),
    ("RAG-Sequence (DPR + BART)",   "MS-MARCO", "BLEU-1",  44.2),
]
for model, dataset, metric, score in paper:
    print(f"{model:<32} {dataset:<12} {metric:<10} {score:>7.1f}")

print("-" * 72)

# Our results (filled in from Cells 9 and 12)
ours = [
    ("ColBERT + LLaMA (ours)",      "NQ",       "EM",      em_score),
    ("ColBERT + LLaMA (ours)",      "MS-MARCO", "ROUGE-L", rouge_l),
    ("ColBERT + LLaMA (ours)",      "MS-MARCO", "BLEU-1",  bleu1),
]
for model, dataset, metric, score in ours:
    print(f"{model:<32} {dataset:<12} {metric:<10} {score:>7.2f}")

print("=" * 72)